# Your Own Data: LAS, formations, lithology and diagnostics

Start with LAS alone, then add only the geology you actually have. This self-contained notebook uses bundled data and writes temporary demonstration CSV/TSV/Excel files; it does not modify package datasets.

**Beginner path:** sections 1-3. **Advanced path:** column/sheet/well mapping, depth metadata, explicit terminal depths, source selection and error handling in sections 4-8.

These are narrow tabular exchange profiles, not a universal industry format or a WITSML/PDF/mixture importer. See the [input guide](../docs/input_templates.rst) for templates and industry-source references.

In [ ]:
%matplotlib inline
from pathlib import Path
from tempfile import TemporaryDirectory
import numpy as np
import pandas as pd
import petrophys as pp
from IPython.display import display

scratch = TemporaryDirectory(prefix="petrophys-notebook-")
workdir = Path(scratch.name)

## 1. LAS alone is sufficient for a first plot

Replace the sample path with your own LAS path. Plotting does not require formations, lithology, Rw, or a completed interpretation. Verify the well identity and depth unit first.

In [ ]:
raw = pp.read(pp.sample("30_6-5"))
print(raw.name, raw.top, raw.bottom, raw.depth_unit)
viewer = pp.plot(raw, depth_range=(2800, 3000), figsize=(16, 9))
viewer.show()
viewer.close()
print(viewer.warnings)

## 2. Inspect the templates before filling them

Formation **tops** are point picks; formation **intervals** include actual bottoms. Lithology intervals require top, bottom and a rock label. All template rows below are **synthetic**, not interpretations of the bundled well.

Use CSV UTF-8 with one header row and numeric depth cells. Optional metadata can identify the well, length unit, depth reference and datum. Gaps in lithology remain gaps; overlaps and reversed intervals are rejected.

In [ ]:
for name in ("formation_tops_template", "formation_intervals_template",
             "lithology_intervals_template", "lithology_codes_template"):
    print(name)
    display(pd.read_csv(pp.sample(name)))

## 3. Attach either or both optional geology tables

For a reproducible real-well example, use the bundled 30/6-5 companions, not the synthetic templates. Remove either keyword when that information is unavailable. Attached external lithology takes precedence in the default workflow; passing intervals only to a plot does not change interpretation inputs.

In [ ]:
well = pp.read(
    pp.sample("30_6-5"),
    formations=pp.sample("formations_30_6-5"),
    lithology=pp.sample("lithology_30_6-5"),
)
viewer = pp.plot(well, depth_range=(2800, 3000), figsize=(16, 9))
viewer.show()
viewer.close()
display(viewer.diagnostics)

## 4. Map real-world table layouts explicitly

A supplier may export one Excel workbook with several sheets and wells. The following is a **constructed vendor-layout example**, using two known tops from this well plus one deliberately unrelated well.

The source column names differ from the template, and depths are expressed in feet. We declare MD and RKB on both sides of the join for this example. In a real delivery, confirm those declarations with the source data; setting metadata is not a survey or datum transformation.

Column mappings run from canonical field name to source header. Select both the intended worksheet and well. The importer reports filtered rows and compatible unit conversions.

In [ ]:
reference_well = pp.read(pp.sample("30_6-5"), depth_reference="MD", datum="RKB")
supplier = pd.DataFrame({
    "Well ID": ["30/6-5", "30/6-5", "OTHER-WELL"],
    "Marker": ["Ness Fm.", "Etive Fm.", "Unrelated"],
    "Pick feet": np.array([2841.0, 2865.0, 1000.0]) / 0.3048,
})
xlsx = workdir / "supplier_tops.xlsx"
with pd.ExcelWriter(xlsx) as writer:
    supplier.to_excel(writer, sheet_name="Tops", index=False)
    pd.DataFrame({"note": ["Demonstration workbook"]}).to_excel(writer, sheet_name="Notes", index=False)

mapping = {"well": "Well ID", "formation_name": "Marker", "top_depth": "Pick feet"}
tops = pp.read_formations(
    xlsx, sheet_name="Tops", well=reference_well, columns=mapping,
    depth_unit="ft", depth_reference="MD", datum="RKB",
)
np.testing.assert_allclose(tops.top_depth, [2841.0, 2865.0])
assert tops.attrs["import_report"]["filtered_rows"] == 1
display(tops)
display(tops.attrs["import_report"])

### Semicolon text and detailed lithology names

This example reformats the bundled lithology table, retaining its original rock names. An explicit mapping allows vendor comment columns to be omitted; the import report lists them. Numeric vendor codes must first be decoded with a verified codebook, not guessed to use the GeoLink scheme.

In [ ]:
original_lithology = pp.read_lithology_intervals(pp.sample("lithology_30_6-5"))
supplier_lithology = original_lithology.rename(
    columns={"top": "From MD", "bottom": "To MD", "lithology": "Rock"}
)
supplier_lithology["Comment"] = "Demonstration export"
text_path = workdir / "supplier_lithology.txt"
supplier_lithology.to_csv(text_path, sep=";", index=False)
lithology = pp.read_lithology_intervals(
    text_path, sep=";", well=reference_well,
    columns={"top": "From MD", "bottom": "To MD", "lithology": "Rock"},
    depth_unit="m", depth_reference="MD", datum="RKB",
)
assert lithology.lithology.tolist() == original_lithology.lithology.tolist()
display(lithology.head())
display(lithology.attrs["import_report"])

### CSV and TSV well logs

Tabular log files need an explicit depth column and unit. The two files below contain a small excerpt of the LAS, not a second independent well. This does not transfer the LAS's full header or interpretation metadata. DLIS uses `pp.read` too, with the optional `dlis` dependency; borehole images have their own specialist reader.

In [ ]:
excerpt = pd.DataFrame({"MD": raw.depth[:20], "GR": raw["GR"].data[:20]})
for suffix, sep in ((".csv", ","), (".tsv", "\t")):
    path = workdir / ("log_excerpt" + suffix)
    excerpt.to_csv(path, index=False, sep=sep)
    imported = pp.read(path, depth_column="MD", depth_unit=raw.depth_unit)
    np.testing.assert_allclose(imported.depth, excerpt.MD)
    print(suffix, imported.n_samples, imported.depth_unit)

## 5. Distinguish picks from bounded intervals

The viewer can draw `tops` directly without inventing bottoms. Analysis needs bounded intervals. Supply an actual terminal boundary, here 2958 m for the example Ness/Etive interval. There is no automatic '+100 units' final formation.

Mixed ranks or coincident tops need explicit bottoms or rank selection before deriving intervals. The formation summary API aggregates the covered depth union; do not sum overlapping group/member rows yourself.

In [ ]:
from petrophys.model.formation_tops import FormationTops

bounded = FormationTops.from_dataframe(tops, terminal_depth=2958.0)
display(bounded.to_dataframe())
assert bounded.intervals[-1].bottom_depth == 2958.0
print("Rank and nested-total examples are in lesson 11.")

## 6. Interpret with explicit assumptions and source selection

`Rw = 0.070 ohm.m at 230 F` below is a teaching assumption. Review the fluid, temperature, matrix and cutoff choices before interpreting another well.

The default source policy is external intervals, then a recognized vendor log, then computed lithology. Alternatives are `lithology_source="vendor"`, `"computed"` or `"config"`; config requires explicit matrix/lithology choices. Missing explicitly requested inputs fail.

No external geology is required for the normal workflow. The mineral solver runs only when supported by its logs. See lesson 05 for the no-geology solver route.

In [ ]:
curves_before = tuple(well.curves)
result = pp.interpret(well, rw=0.070, rw_meas_temp_f=230.0)
result.raise_for_errors()
assert result.well is not well
assert tuple(well.curves) == curves_before

context_step = next(step for step in result.steps_executed if step.step == "input_context")
assert context_step.parameters["selected_source"] == "external"
print("Status:", result.status)
display(context_step.parameters)
print("Skipped:", result.specific_steps_skipped)
print("Warnings:", result.warnings)
display(result.to_dataframe()[["Step", "Status", "Method"]])

## 7. Core saturation and calculated SW are different products

The display and workflow normalize recognized core percentages to fractions. For the bundled LAS, percent-like values conflict with the 'frac' header; the conversion is inferred and reported, not treated as proof of units.

Inspect the stored conversion evidence. On an original, unnormalized curve, `standardize_core_saturation(curve, source_unit="%")` can explicitly declare percentages; `source_unit="v/v"` confirms fractions. Unknown scales cannot always be inferred. Neither a unit conversion nor a visual match establishes that core measurements equal in-situ log-derived saturation.

In [ ]:
core = result.well.get("SWC")
if core is not None:
    display(core.metadata.custom.get("core_saturation_normalization", {}))
    finite = core.data[np.isfinite(core.data)]
    assert np.all((finite >= 0) & (finite <= 1))
    print("Core fraction range:", finite.min(), finite.max())

viewer = result.plot(depth_range=(2800, 3000), figsize=(16, 9))
viewer.show()
viewer.close()

## 8. Diagnose input problems before trusting a plot

The examples below deliberately fail and check the actual errors. They are not skipped cells. Notice that a sheet choice is required and a TVDSS-to-MD join is rejected, even when the numeric depths overlap.

Missing depth metadata is recorded as an assumption. Numeric overlap alone is not evidence of correct physical alignment.

In [ ]:
def show_expected_error(label, action):
    try:
        action()
    except (ValueError, FileNotFoundError) as exc:
        print(label + ":", exc)
    else:
        raise AssertionError("Expected validation error: " + label)

show_expected_error("Missing file", lambda: pp.read_formations(workdir / "missing.csv"))
show_expected_error("Choose a sheet", lambda: pp.read_formations(xlsx, columns=mapping))
show_expected_error(
    "Reference mismatch",
    lambda: pp.read_formations(
        supplier.iloc[:2], columns=mapping, well=reference_well,
        depth_unit="ft", depth_reference="TVDSS", datum="RKB",
    ),
)
show_expected_error(
    "Overlapping lithology",
    lambda: pp.read_lithology_intervals(pd.DataFrame({
        "top": [1000, 1050], "bottom": [1100, 1150], "lithology": ["Sandstone", "Shale"],
    })),
)

## Save an output and keep the trace

Use `viewer.save` for figures and `result.to_trace_dict` for machine-readable decisions. This demonstration writes inside its temporary directory and then removes those temporary files. For a real project, choose your own output directory and retain source files, import metadata and assumptions.

In [ ]:
import json

export_viewer = result.plot(depth_range=(2800, 3000), figsize=(16, 9))
figure_path = workdir / "interpretation.png"
export_viewer.save(figure_path, dpi=120)
export_viewer.close()
trace_path = workdir / "workflow_trace.json"
trace_path.write_text(json.dumps(result.to_trace_dict(), indent=2), encoding="utf-8")
assert figure_path.stat().st_size > 0
assert json.loads(trace_path.read_text(encoding="utf-8"))["status"] == result.status
print("Figure and trace export verified.")
scratch.cleanup()

## Next steps

- [Quickstart](quickstart.ipynb): the shortest read/plot/interpret route.
- [QA/QC](part_a_foundations/02_qaqc_and_standardization.ipynb): core unit overrides and provenance.
- [Workflow](part_b_advanced/10_dynamic_workflow_glass_box.ipynb): advanced configuration and traces.
- [Formation analysis](part_b_advanced/11_formation_and_interval_analysis.ipynb): hierarchy, explicit bottoms and union totals.
- [Visualization](part_b_advanced/17_visualization_masterclass.ipynb): track selection and specialist figures.

**Exercise:** remove both geology arguments from section 3 and rerun. Then add only formations, then only lithology. Compare the plot diagnostics and `input_context` trace rather than assuming every run selected the same source.